# JEPA Writeup 7/22 results


In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'results').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the cody-jepa repository root')


REPO_ROOT = find_repo_root()
HISTORY_PATH = REPO_ROOT / 'results' / 'checkpoint_histories.csv'
SUMMARY_PATH = REPO_ROOT / 'results' / 'phase0_summary.json'
OUTPUT_DIR = Path(os.environ.get(
    'CODY_JEPA_REPRO_OUTPUT_DIR',
    REPO_ROOT / 'results' / 'generated' / 'writeup-7-22',
)).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for required in (HISTORY_PATH, SUMMARY_PATH):
    if not required.is_file():
        raise FileNotFoundError(required)

print(f'Repository: {REPO_ROOT}')
print(f'History: {HISTORY_PATH.relative_to(REPO_ROOT)}')
print(f'Generated artifacts: {OUTPUT_DIR}')


## Load and validate stored values

Training loss and cosine are recorded every epoch. Validation metrics are recorded every five epochs. No values are interpolated or transcribed from the PDF.


In [ ]:
history = pd.read_csv(HISTORY_PATH, float_precision='round_trip')
history = history.loc[history['run_id'].eq('phase0-job-91108')].copy()
if len(history) != 100:
    raise ValueError('Expected a complete 100-epoch history')
if int(history['epoch'].max()) != 100 or int(history['step'].max()) != 3900:
    raise ValueError('Expected the completed epoch-100, step-3900 checkpoint')

values = history.rename(columns={
    'val_loss': 'validation_loss',
    'val_cosine': 'validation_cosine',
    'val_feature_std': 'feature_std',
    'val_effective_rank': 'effective_rank',
    'val_effective_rank_ratio': 'effective_rank_ratio',
    'val_near_zero_variance_fraction': 'near_zero_variance_fraction',
    'val_subject_balanced_context_shuffle_loss_gap': 'wrong_context_gap',
})[[
    'epoch', 'step', 'train_loss', 'train_cosine', 'validation_loss',
    'validation_cosine', 'feature_std', 'effective_rank',
    'effective_rank_ratio', 'near_zero_variance_fraction', 'wrong_context_gap',
]].copy()
evaluations = values.dropna(subset=['validation_loss']).copy()
if len(evaluations) != 20 or evaluations['epoch'].tolist() != list(range(5, 101, 5)):
    raise ValueError('Expected validation metrics at every fifth epoch')
if not np.isfinite(values[['train_loss', 'train_cosine']].to_numpy()).all():
    raise ValueError('Training history contains non-finite values')
if not np.isfinite(evaluations.drop(columns=['epoch', 'step']).to_numpy()).all():
    raise ValueError('Validation history contains non-finite values')

values_path = OUTPUT_DIR / 'writeup-7-22-plot-values.csv'
values.to_csv(values_path, index=False, float_format='%.17g')
print(f'Validated {len(values)} training rows and {len(evaluations)} validation rows')
print(f'Exact plotted values: {values_path}')


In [ ]:
def require_close(label: str, observed: float, expected: float) -> None:
    if not np.isclose(observed, expected, rtol=1e-10, atol=1e-12):
        raise AssertionError(f'{label}: outputs={observed!r}, report={expected!r}')


report = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
metric_map = {
    'validation_loss': 'validation_loss',
    'effective_rank': 'effective_rank',
    'effective_rank_ratio': 'effective_rank_ratio',
    'feature_std': 'feature_std',
    'near_zero_variance_fraction': 'near_zero_variance_fraction',
    'wrong_context_gap': 'wrong_context_gap',
}
for checkpoint_summary in report['checkpoints']:
    epoch = int(checkpoint_summary['epoch'])
    observed = evaluations.loc[evaluations['epoch'].eq(epoch)].iloc[0]
    for report_key, frame_key in metric_map.items():
        require_close(f"{checkpoint_summary['label']} {report_key}", observed[frame_key], checkpoint_summary[report_key])

print(f'All plotted history metrics match {SUMMARY_PATH.relative_to(REPO_ROOT)}')


## Six-panel result figure


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

axes[0, 0].plot(values['step'], values['train_loss'], label='train')
axes[0, 0].plot(evaluations['step'], evaluations['validation_loss'], 'o-', label='validation')
axes[0, 0].set_title('JEPA loss')

axes[0, 1].plot(values['step'], values['train_cosine'], label='train')
axes[0, 1].plot(evaluations['step'], evaluations['validation_cosine'], 'o-', label='validation')
axes[0, 1].set_title('Cosine similarity')

axes[0, 2].plot(evaluations['step'], evaluations['feature_std'], 'o-')
axes[0, 2].set_title('Online full-view feature std')

axes[1, 0].plot(evaluations['step'], evaluations['effective_rank'], 'o-')
axes[1, 0].set_title('Online full-view effective rank')

axes[1, 1].plot(evaluations['step'], evaluations['near_zero_variance_fraction'], 'o-')
axes[1, 1].set_title('Online full-view near-zero variance fraction')

axes[1, 2].plot(evaluations['step'], evaluations['wrong_context_gap'], 'o-')
axes[1, 2].axhline(0, color='black', linewidth=1)
axes[1, 2].set_title('Subject-balanced wrong-context loss gap')

for axis in axes.flat:
    axis.set_xlabel('optimizer step')
    axis.xaxis.set_major_locator(MaxNLocator(8, integer=True))
    axis.grid(True, alpha=0.3)
    if axis.get_legend_handles_labels()[0]:
        axis.legend(frameon=False)

figure_path = OUTPUT_DIR / 'writeup-7-22-health.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print(figure_path)


In [ ]:
best = evaluations.loc[evaluations['validation_loss'].idxmin()]
final = evaluations.iloc[-1]
headline = pd.DataFrame([
    {'quantity': 'best validation loss', 'epoch': int(best['epoch']), 'value': best['validation_loss']},
    {'quantity': 'final effective rank', 'epoch': int(final['epoch']), 'value': final['effective_rank']},
    {'quantity': 'final effective-rank ratio', 'epoch': int(final['epoch']), 'value': final['effective_rank_ratio']},
    {'quantity': 'final feature std', 'epoch': int(final['epoch']), 'value': final['feature_std']},
    {'quantity': 'final near-zero variance fraction', 'epoch': int(final['epoch']), 'value': final['near_zero_variance_fraction']},
    {'quantity': 'final wrong-context gap', 'epoch': int(final['epoch']), 'value': final['wrong_context_gap']},
])
if int(best['epoch']) != 80:
    raise AssertionError('The stored history no longer has its best validation loss at epoch 80')
print(headline.to_string(index=False))
